In [91]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

In [92]:
# 1. 데이터 로드
store_info = pd.read_csv("./data4/realistic_store_info_20.csv")
daily_cashflow = pd.read_csv("./data4/realistic_daily_cashflow_20.csv")
fixed_cost = pd.read_csv("./data4/realistic_fixed_cost_20.csv")
loan_info = pd.read_csv("./data4/realistic_loan_info_20.csv")

In [93]:
# 2. 날짜형 변환
# 날짜: 문자열 -> 시간 데이터 타입 (datetime)
daily_cashflow["date"] = pd.to_datetime(daily_cashflow["date"])
fixed_cost["month"] = pd.to_datetime(fixed_cost["month"] + "-01")
loan_info["loan_start_date"] = pd.to_datetime(loan_info["loan_start_date"])

In [94]:
# 3. 정렬
daily_cashflow = daily_cashflow.sort_values(["store_id", "date"]).reset_index(drop=True)
fixed_cost = fixed_cost.sort_values(["store_id", "month"]).reset_index(drop=True)
loan_info = loan_info.sort_values(["store_id", "loan_start_date"]).reset_index(drop=True)

In [95]:
# 4. 기본 전처리
expense_cols = [
    "platform_fee",
    "ingredient_cost",
    "labor_cost",
    "utilities_cost",
    "marketing_cost",
    "other_expense",
    "loan_repayment",
]

daily_cashflow["daily_revenue"] = daily_cashflow["total_sales"]
daily_cashflow["daily_expense"] = daily_cashflow[expense_cols].sum(axis=1)
daily_cashflow["month"] = daily_cashflow["date"].values.astype("datetime64[M]")

# 월별 매출
monthly_sales = (
    daily_cashflow.groupby(["store_id", "month"], as_index=False)["total_sales"]
    .sum()
    .rename(columns={"total_sales": "monthly_sales"})
)

# 월별 대출 상환액
monthly_loan = (
    loan_info.groupby("store_id", as_index=False)["monthly_payment"]
    .sum()
    .rename(columns={"monthly_payment": "total_monthly_payment"})
)

# 고정비 증가율
fixed_cost["fixed_cost_3m_ago"] = fixed_cost.groupby("store_id")["fixed_cost_total"].shift(3)
fixed_cost["fixed_cost_growth_rate"] = np.where(
    fixed_cost["fixed_cost_3m_ago"] > 0,
    (fixed_cost["fixed_cost_total"] - fixed_cost["fixed_cost_3m_ago"]) / fixed_cost["fixed_cost_3m_ago"],
    0.0
)
fixed_cost["fixed_cost_growth_rate"] = (
    fixed_cost["fixed_cost_growth_rate"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

In [96]:
# 5. 데이터 결합
df = daily_cashflow.merge(
    fixed_cost[["store_id", "month", "fixed_cost_total", "fixed_cost_growth_rate"]],
    on=["store_id", "month"],
    how="left"
)

df = df.merge(monthly_sales, on=["store_id", "month"], how="left")
df = df.merge(monthly_loan, on="store_id", how="left")

# 결측 보정
df["fixed_cost_total"] = df["fixed_cost_total"].fillna(df["fixed_cost_total"].median())
df["fixed_cost_growth_rate"] = df["fixed_cost_growth_rate"].fillna(0)
df["monthly_sales"] = df["monthly_sales"].fillna(0)
df["total_monthly_payment"] = df["total_monthly_payment"].fillna(0)

df = df.sort_values(["store_id", "date"]).reset_index(drop=True)

In [97]:
# 6. 파생변수 생성
g = df.groupby("store_id")

df["sales_7d_avg"] = g["total_sales"].transform(lambda x: x.rolling(7, min_periods=1).mean())
df["sales_30d_avg"] = g["total_sales"].transform(lambda x: x.rolling(30, min_periods=1).mean())
df["avg_daily_expense_30"] = g["daily_expense"].transform(lambda x: x.rolling(30, min_periods=1).mean())
df["avg_net_cashflow_30"] = g["net_cashflow"].transform(lambda x: x.rolling(30, min_periods=1).mean())
df["std_net_cashflow_30"] = g["net_cashflow"].transform(lambda x: x.rolling(30, min_periods=3).std())

# sales_30d_avg를 avg_daily_sales_30로 재사용
df["avg_daily_sales_30"] = df["sales_30d_avg"]

df["sales_trend_factor"] = np.where(
    df["sales_30d_avg"] > 0,
    df["sales_7d_avg"] / df["sales_30d_avg"],
    1.0
)
df["sales_trend_factor"] = (
    df["sales_trend_factor"]
    .replace([np.inf, -np.inf], 1.0)
    .fillna(1.0)
    .clip(0.5, 1.5)
)

df["curr_30d_sales"] = g["total_sales"].transform(lambda x: x.rolling(30, min_periods=1).sum())
df["prev_30d_sales"] = g["curr_30d_sales"].shift(30)

df["sales_decline_rate"] = np.where(
    df["prev_30d_sales"] > 0,
    (df["prev_30d_sales"] - df["curr_30d_sales"]) / df["prev_30d_sales"],
    0.0
)
df["sales_decline_rate"] = (
    df["sales_decline_rate"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

df["loan_burden_ratio"] = np.where(
    df["monthly_sales"] > 0,
    df["total_monthly_payment"] / df["monthly_sales"],
    0.0
)
df["loan_burden_ratio"] = (
    df["loan_burden_ratio"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

# 달력 피처
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek
df["month_num"] = df["date"].dt.month
df["days_in_month"] = df["date"].dt.days_in_month
df["days_remaining"] = df["days_in_month"] - df["day"]

In [98]:
# 7. 실제 월말 잔액
actual_month_end = (
    df.groupby(["store_id", "month"], as_index=False)
      .tail(1)[["store_id", "month", "closing_balance"]]
      .rename(columns={"closing_balance": "actual_end_balance"})
)

df = df.merge(actual_month_end, on=["store_id", "month"], how="left")

### 왜 LightGBM ?
* 복잡한 비선형 관계를 LightGBM이 잘 잡아주기 때문
* 모델은 단순히 '현재 잔액'만 보는 것이 아닌 '현재 잔액 + 최근 매출 흐름 + 지출 흐름 + 고정비 부담 + 월말까지 남은 날짜'를 함께 보고 월말 잔액 예측
* actual_end_balance: 매장별·월별로 마지막 날짜의 closing_balance
* predicted_end_balance: 각 날짜 시점에서 모델이 예상한 그 달 말의 최종 잔액

In [99]:
# 8. LightGBM 월말 잔액 예측
feature_cols = [
    "closing_balance",
    "daily_revenue",
    "daily_expense",
    "net_cashflow",
    "sales_7d_avg",
    "sales_30d_avg",
    "avg_daily_sales_30",
    "avg_daily_expense_30",
    "avg_net_cashflow_30",
    "std_net_cashflow_30",
    "sales_trend_factor",
    "sales_decline_rate",
    "fixed_cost_total",
    "fixed_cost_growth_rate",
    "monthly_sales",
    "total_monthly_payment",
    "loan_burden_ratio",
    "day",
    "dayofweek",
    "month_num",
    "days_remaining",
]

for c in feature_cols:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

train_mask = df["date"] < pd.Timestamp("2025-10-01")
test_mask = df["date"] >= pd.Timestamp("2025-10-01")

X_train = df.loc[train_mask, feature_cols]
y_train = df.loc[train_mask, "actual_end_balance"]

X_test = df.loc[test_mask, feature_cols]
y_test = df.loc[test_mask, "actual_end_balance"]

model = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
)

model.fit(X_train, y_train)

df["predicted_end_balance"] = model.predict(df[feature_cols])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000754 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3753
[LightGBM] [Info] Number of data points in the train set: 5460, number of used features: 21
[LightGBM] [Info] Start training from score 47020828.716117
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

### 위험 점수 계산 방법
##### 1) short risk
* 단기 위험, 즉 최근 30일 기준 위험 점수
* 3개 요소로 계산 
    * 현금 버팀력 위험(잔액 생존기간)
        * 월말 잔액 / 월 예상 지출
    * 수익성 위험
        * 30일 평균 매출 / 30일 평균 지출
    * 적자 빈도 위험
        * daily_revenue < daily_expense인 날의 비율

##### 2) mid risk
* 중기 위험, 즉 최근 90일 기준 위험 점수
* 원리는 short risk와 동일. 다만 30일 대신 90일 평균, 비율 사용

##### short risk와 mid risk를 통해 알 수 있는 점
* short risk: 최근 급한 이상신호
* mid_risk: 지속적/구조적 위험

In [102]:
def restaurant_cashflow_risk_model(df):
    df = df.copy().sort_values(["store_id", "date"]).reset_index(drop=True)
    g = df.groupby("store_id")

    df["rev_30d"] = g["daily_revenue"].transform(lambda x: x.rolling(30, min_periods=10).mean())
    df["exp_30d"] = g["daily_expense"].transform(lambda x: x.rolling(30, min_periods=10).mean())

    df["rev_90d"] = g["daily_revenue"].transform(lambda x: x.rolling(90, min_periods=30).mean())
    df["exp_90d"] = g["daily_expense"].transform(lambda x: x.rolling(90, min_periods=30).mean())

    deficit = (df["daily_revenue"] < df["daily_expense"]).astype(int)
    df["deficit_ratio_30d"] = deficit.groupby(df["store_id"]).transform(
        lambda x: x.rolling(30, min_periods=10).mean()
    )
    df["deficit_ratio_90d"] = deficit.groupby(df["store_id"]).transform(
        lambda x: x.rolling(90, min_periods=30).mean()
    )

    def runway_risk(x):
        if pd.isna(x):
            return np.nan
        if x >= 4:
            return 0
        elif x >= 2.5:
            return (4 - x) / 1.5
        elif x >= 1.5:
            return 0.5 + (2.5 - x)
        else:
            return 1

    def margin_risk(x):
        if pd.isna(x):
            return np.nan
        if x >= 1.2:
            return 0
        elif x >= 1.05:
            return (1.2 - x) / 0.15
        elif x >= 1:
            return 0.5 + (1.05 - x) / 0.05
        else:
            return 1

    def deficit_risk(x):
        if pd.isna(x):
            return np.nan
        if x < 0.1:
            return 0
        elif x < 0.25:
            return (x - 0.1) / 0.15
        elif x < 0.4:
            return 0.5 + (x - 0.25) / 0.15
        else:
            return 1

    df["runway_months_30d"] = df["predicted_end_balance"] / np.maximum(df["exp_30d"] * 30, 1)
    df["margin_ratio_30d"] = df["rev_30d"] / np.maximum(df["exp_30d"], 1)

    df["short_risk"] = (
        0.5 * df["runway_months_30d"].apply(runway_risk) +
        0.3 * df["margin_ratio_30d"].apply(margin_risk) +
        0.2 * df["deficit_ratio_30d"].apply(deficit_risk)
    ) * 100

    df["runway_months_90d"] = df["predicted_end_balance"] / np.maximum(df["exp_90d"] * 30, 1)
    df["margin_ratio_90d"] = df["rev_90d"] / np.maximum(df["exp_90d"], 1)

    df["mid_risk"] = (
        0.5 * df["runway_months_90d"].apply(runway_risk) +
        0.3 * df["margin_ratio_90d"].apply(margin_risk) +
        0.2 * df["deficit_ratio_90d"].apply(deficit_risk)
    ) * 100

    df["total_risk"] = 0.6 * df["short_risk"] + 0.4 * df["mid_risk"]

    df["short_risk"] = df["short_risk"].clip(0, 100)
    df["mid_risk"] = df["mid_risk"].clip(0, 100)
    df["total_risk"] = df["total_risk"].clip(0, 100)

    df["rev_3d"] = g["daily_revenue"].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
    df["exp_3d"] = g["daily_expense"].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
    df["exp_7d"] = g["daily_expense"].transform(
        lambda x: x.rolling(7, min_periods=3).mean()
    )

    deficit_daily = (df["daily_revenue"] < df["daily_expense"]).astype(int)
    df["deficit_ratio_7d"] = deficit_daily.groupby(df["store_id"]).transform(
        lambda x: x.rolling(7, min_periods=3).mean()
    )

    df["runway_days_1d"] = df["closing_balance"] / np.maximum(df["exp_7d"], 1)
    df["runway_months_1d"] = df["runway_days_1d"] / 30

    df["margin_ratio_1d"] = df["rev_3d"] / np.maximum(df["exp_3d"], 1)

    df["daily_risk"] = (
        0.5 * df["runway_months_1d"].apply(runway_risk) +
        0.3 * df["margin_ratio_1d"].apply(margin_risk) +
        0.2 * df["deficit_ratio_7d"].apply(deficit_risk)
    ) * 100

    df["daily_risk"] = df["daily_risk"].clip(0, 100)

    def daily_risk_level_func(x):
        if pd.isna(x):
            return "데이터부족"
        elif x < 25:
            return "안정"
        elif x < 50:
            return "주의"
        elif x < 75:
            return "위험"
        else:
            return "심각"

    df["daily_risk_level"] = df["daily_risk"].apply(daily_risk_level_func)

    def risk_level(x):
        if pd.isna(x):
            return "데이터부족"
        elif x < 25:
            return "안정"
        elif x < 50:
            return "주의"
        elif x < 75:
            return "위험"
        else:
            return "심각"

    df["risk_level"] = df["total_risk"].apply(risk_level)

    result = df[
        [
            "store_id",
            "date",
            "daily_revenue",
            "daily_expense",
            "closing_balance",
            "predicted_end_balance",
            "actual_end_balance",
            "daily_risk",
            "daily_risk_level",
            "short_risk",
            "mid_risk",
            "total_risk",
            "risk_level",
        ]
    ].copy()

    round_cols = [
        "predicted_end_balance",
        "actual_end_balance",
        "daily_risk",
        "short_risk",
        "mid_risk",
        "total_risk",
    ]

    for col in round_cols:
        result[col] = pd.to_numeric(result[col], errors="coerce").round(2)

    return result

In [104]:
# 10. 최종 결과 저장
result = restaurant_cashflow_risk_model(df)
result.to_csv("risk_prediction_result_final.csv", index=False, encoding="utf-8-sig")

print(result.tail(10))
print(result.columns.tolist())
print(result["risk_level"].value_counts(dropna=False))

     store_id       date  daily_revenue  daily_expense  closing_balance  \
7290     S020 2025-12-22        1218941         683582        115205119   
7291     S020 2025-12-23        1013312         565257        115653174   
7292     S020 2025-12-24         680351         405754        115927771   
7293     S020 2025-12-25        1091254        1777602        115241423   
7294     S020 2025-12-26        2427661        1748606        115920478   
7295     S020 2025-12-27        1799443        1027054        116692867   
7296     S020 2025-12-28        1135912         658347        117170432   
7297     S020 2025-12-29         754125         465785        117458772   
7298     S020 2025-12-30        1188121         988953        117657940   
7299     S020 2025-12-31         887905         522098        118023747   

      predicted_end_balance  actual_end_balance  daily_risk daily_risk_level  \
7290           1.030967e+08           118023747        0.00               안정   
7291          